# FuzzyDiff on SD3-Medium

Settings: GPU T4 x2, Internet on. Add a Kaggle Secret named `HF_TOKEN`, from a Hugging Face account that has accepted the SD3-Medium licence.

All logic lives in `code/`. Results from this notebook use provisional calibration: they show the pipeline runs and are never paper evidence.


In [ ]:
BRANCH = 'fuzzysd3'
OUT = '/kaggle/working/outputs'
RESUME = ''  # path to an earlier session's outputs, attached as a Kaggle Dataset
VAE_DEVICE = 'cuda:1'  # the transformer stays on cuda:0

!git clone -q --branch {BRANCH} https://github.com/sunzidulislam/fuzzydiff.git /kaggle/working/fuzzydiff
%cd /kaggle/working/fuzzydiff
!pip install -q -r requirements-kaggle.txt
!git log -1 --oneline


In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
!nvidia-smi -L


## 1. Verify before anything else

`test_processor.py` runs on CPU. `verify_checkpoint.py` checks the real weights and reports memory: look at `mem_guided_gb` before going on. Its output goes to `/kaggle/working/verify`.


In [ ]:
!python -I code/test_processor.py | tail -5
!mkdir -p /kaggle/working/verify
!python -I code/verify_checkpoint.py --model stabilityai/stable-diffusion-3-medium-diffusers --out /kaggle/working/verify | tail -30


## 2. M1 tuning

Three step sizes × two guided windows × two directions on 5 separate prompts.


In [ ]:
resume = f'--resume-from {RESUME}/m1_tune' if RESUME else ''
!python -I code/run_experiment.py --suite m1_tune --out {OUT}/m1_tune --vae-device {VAE_DEVICE} {resume}


Set the chosen `cfgEtaZero` and `cfgSigmaLo` in `configs/provisional.yaml` before the next cell.


In [ ]:
import json, glob, collections
rows = collections.defaultdict(list)
for p in glob.glob(f'{OUT}/m1_tune/*@*/*/meta.json'):
    m = json.load(open(p))
    if 'dy' in m:
        rows[m['arm']].append((m['ds'], m['dy'], m['time_s']))
for arm, r in sorted(rows.items()):
    print(arm, 'n', len(r), 'mean dy/ds', round(sum(b / a for a, b, _ in r) / len(r), 3), 'time_s', round(sum(t for *_, t in r) / len(r), 1))


## 3. M1 evaluation and ladder


In [ ]:
resume = f'--resume-from {RESUME}/m1' if RESUME else ''
!python -I code/run_experiment.py --suite m1_eval --out {OUT}/m1 --vae-device {VAE_DEVICE} {resume}


## 4. Blind rating page

Download `rate.html`, rate on your own machine, and save the exported text as `ratings.json`. Don't open `rate.key.json` until you've finished rating.


In [ ]:
import sys, glob
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import build_rating_page
images = [Path(p) for p in glob.glob(f'{OUT}/m1/*/*/image.png') if '/ladder' not in p]
build_rating_page(images, Path(f'{OUT}/m1/rate.html'), seed=0)
print(len(images), 'images')
